# 08 — RAG Evaluation: Retrieval Precision/Recall and LLM Stance-Label Spot-Check

Before this notebook, retrieval and generation quality were judged by eyeballing individual examples (see [decoded-state-to-text-report.md](../docs/decoded-state-to-text-report.md)). This builds a small, honestly-labeled evaluation set — reading each of the 6 corpus papers to make real relevance judgments, not guessing — and measures:

1. **Retrieval precision/recall** at k=5, dense-only vs. dense+cross-encoder-rerank.
2. **LLM stance-label spot-check** — does `pipeline.py`'s SUPPORTS/CONTRADICTS/UNRELATED labeling match manual judgment on a couple of examples.

**Caveat up front**: 5 near-identical queries (one per MOTOR condition) against 6 papers is a small, first-pass evaluation, not a rigorous benchmark — good enough to catch an obviously broken reranker or a wildly hallucinating LLM, not to certify quality.


In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NeuroLens repository root.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurolens.retrieval import load_index, load_embedding_model, load_reranker, retrieve_chunks, retrieve_and_rerank
from neurolens.pipeline import build_query_text

chunks, embeddings = load_index(PROJECT_ROOT / "artifacts" / "paper_index")
embedding_model = load_embedding_model()
reranker = load_reranker()
print(f"{len(chunks)} chunks from {len(set(c.source_file for c in chunks))} papers")


/Users/srinivasgovindasurampudi/miniconda3/envs/neurolens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6369.81it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8425.04it/s]

729 chunks from 6 papers


## 1. Gold relevance judgments (read, not guessed)

Read page 1 of every paper in the corpus to ground these judgments in actual content, not filenames:

- **Barch et al. 2013** (`1-s2.0-S1053811913005272-main.pdf`) — the actual HCP task-fMRI design paper, explicitly describes the MOTOR task (hand/foot/tongue movements). **HIGH relevance** to any MOTOR-condition query.
- **Yeo et al. 2011** (`thomas-yeo-et-al-2011...pdf`) — defines the 7-network parcellation (SomMot, Default, etc.) named in every query. **Relevant** for grounding the RSN vocabulary, not the task itself.
- **Schaefer et al. 2018** (`bhx179.pdf`) — defines the exact 300-ROI atlas and network assignments this pipeline uses. **Relevant** for the same reason as Yeo 2011.
- **Van Essen et al. 2013** (`1-s2.0-S1053811913005351-main.pdf`) — general HCP project overview, doesn't discuss MOTOR-task specifics or RSNs in depth. **Background relevance only.**
- **van den Heuvel & Sporns 2013** (`PIIS1364661313002167.pdf`) — "Network hubs in the human brain," a connectome graph-theory review unrelated to task decoding or the Yeo/Schaefer network vocabulary. **Not relevant** to these queries.
- **Misra & Surampudi et al. 2021** (`journal.pcbi.1008943.pdf`) — GRU-based decoding of naturalistic movie-viewing fMRI, a different task domain entirely. **Not relevant** to MOTOR-condition queries (methodologically adjacent, topically off).

**Relevant set for all 5 queries below**: `{Barch2013, Yeo2011, Schaefer2018}` (3 of 6 papers).


In [2]:
RELEVANT_PAPERS = {
    "1-s2.0-S1053811913005272-main.pdf",  # Barch 2013
    "thomas-yeo-et-al-2011-the-organization-of-the-human-cerebral-cortex-estimated-by-intrinsic-functional-connectivity.pdf",  # Yeo 2011
    "bhx179.pdf",  # Schaefer 2018
}
N_RELEVANT_PAPERS = len(RELEVANT_PAPERS)

CONDITIONS = ["left_hand", "right_hand", "left_foot", "right_foot", "tongue"]

def fake_rsn_attribution(network: str) -> dict:
    return {"consensus_network": network, "families_agree": True, "top_network_by_method": {"saliency": network}}

eval_queries = [
    build_query_text(condition, 0.95, fake_rsn_attribution("SomMot"), subject_id="101915", task="MOTOR", run="LR")
    for condition in CONDITIONS
]
for q in eval_queries:
    print(q)


Decoded condition: left_hand (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: right_hand (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: left_foot (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: right_foot (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: tongue (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.


## 2. Precision/recall at k=5: dense-only vs. dense+rerank

In [3]:
def precision_recall(retrieved_df: pd.DataFrame) -> dict:
    retrieved_papers = set(retrieved_df["source_file"])
    hit_papers = retrieved_papers & RELEVANT_PAPERS
    n_relevant_chunks = retrieved_df["source_file"].isin(RELEVANT_PAPERS).sum()
    return {
        "precision_at_5": n_relevant_chunks / len(retrieved_df),
        "recall_papers_at_5": len(hit_papers) / N_RELEVANT_PAPERS,
    }


rows = []
for query, condition in zip(eval_queries, CONDITIONS):
    dense = retrieve_chunks(query, model=embedding_model, embeddings=embeddings, chunks=chunks, top_k=5)
    reranked = retrieve_and_rerank(query, model=embedding_model, embeddings=embeddings, chunks=chunks,
                                     reranker=reranker, candidate_k=20, top_k=5)
    dense_metrics = precision_recall(dense)
    rerank_metrics = precision_recall(reranked)
    rows.append({"condition": condition, "method": "dense-only", **dense_metrics})
    rows.append({"condition": condition, "method": "dense+rerank", **rerank_metrics})

eval_df = pd.DataFrame(rows)
eval_df


,condition,method,precision_at_5,recall_papers_at_5
0,left_hand,dense-only,1.0,1.000000
1,left_hand,dense+rerank,1.0,1.000000
2,right_hand,dense-only,1.0,1.000000
3,right_hand,dense+rerank,1.0,1.000000
4,left_foot,dense-only,1.0,0.666667
5,left_foot,dense+rerank,1.0,1.000000
6,right_foot,dense-only,1.0,0.666667
7,right_foot,dense+rerank,1.0,1.000000
8,tongue,dense-only,1.0,0.666667
9,tongue,dense+rerank,1.0,1.000000


In [4]:
summary = eval_df.groupby("method")[["precision_at_5", "recall_papers_at_5"]].mean()
print("Mean across all 5 queries:")
summary


Mean across all 5 queries:


,precision_at_5,recall_papers_at_5
method,,
dense+rerank,1.0,1.0
dense-only,1.0,0.8


## 3. LLM stance-label spot-check (2 examples)

Manually judged against the same gold relevance set above. Real LLM calls (~20s each), kept to 2 examples for wall-clock reasons — genuinely just a spot-check, not a full evaluation.


In [5]:
from neurolens.pipeline import build_llm_prompt, make_mlx_generate_fn

generate_fn = make_mlx_generate_fn()

spot_check_rows = []
for query, condition in zip(eval_queries[:2], CONDITIONS[:2]):
    retrieved = retrieve_chunks(query, model=embedding_model, embeddings=embeddings, chunks=chunks, top_k=5).to_dict(orient="records")
    prompt = build_llm_prompt(query, retrieved)
    t0 = time.time()
    generated = generate_fn(prompt)
    elapsed = time.time() - t0
    print(f"=== {condition} (generated in {elapsed:.1f}s) ===")
    for i, r in enumerate(retrieved, start=1):
        gold = "RELEVANT" if r["source_file"] in RELEVANT_PAPERS else "not relevant"
        print(f"  [{i}] {r['source_file']} (gold: {gold})")
    print()
    print(generated)
    print()
    spot_check_rows.append({"condition": condition, "retrieved": retrieved, "generated_text": generated})


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 4285.73it/s]

=== left_hand (generated in 17.7s) ===
  [1] 1-s2.0-S1053811913005272-main.pdf (gold: RELEVANT)
  [2] 1-s2.0-S1053811913005272-main.pdf (gold: RELEVANT)
  [3] thomas-yeo-et-al-2011-the-organization-of-the-human-cerebral-cortex-estimated-by-intrinsic-functional-connectivity.pdf (gold: RELEVANT)
  [4] bhx179.pdf (gold: RELEVANT)
  [5] thomas-yeo-et-al-2011-the-organization-of-the-human-cerebral-cortex-estimated-by-intrinsic-functional-connectivity.pdf (gold: RELEVANT)

1. Excerpt 1: SUPPORTS the decoded result, as it discusses the activation of the working memory/cognitive control network, which is related to the motor task performed by subject 101915.

Synthesis: The decoded result's focus on the SomMot network, which is involved in working memory and cognitive control, aligns with the literature excerpt's discussion of the activation of this network in various tasks. This suggests that the decoded result is consistent with prior findings.

2. Excerpt 2: UNRELATED to the decoded result,

=== right_hand (generated in 13.9s) ===
  [1] 1-s2.0-S1053811913005272-main.pdf (gold: RELEVANT)
  [2] thomas-yeo-et-al-2011-the-organization-of-the-human-cerebral-cortex-estimated-by-intrinsic-functional-connectivity.pdf (gold: RELEVANT)
  [3] thomas-yeo-et-al-2011-the-organization-of-the-human-cerebral-cortex-estimated-by-intrinsic-functional-connectivity.pdf (gold: RELEVANT)
  [4] bhx179.pdf (gold: RELEVANT)
  [5] 1-s2.0-S1053811913005272-main.pdf (gold: RELEVANT)

1. Excerpt 1: SUPPORTS the decoded result, as it discusses the working memory task and the activation of the right hand, which is consistent with the decoded condition "right_hand" (confidence 95%).

2. Excerpt 2: UNRELATED to the decoded result, as it discusses the organization of the human cerebral cortex and the intrinsic functional connectivity, which is not directly related to the decoded condition "right_hand".

3. Excerpt 3: UNRELATED to the decoded result, as it discusses the precentral cortex and the difficulty i

## Summary

**Retrieval (5 queries, dense-only vs. dense+cross-encoder-rerank, k=5):**

| Method | Precision@5 | Recall (relevant papers)@5 |
|---|---|---|
| Dense-only | 1.00 | 0.80 |
| Dense+rerank | 1.00 | **1.00** |

Precision was already perfect for dense-only retrieval - the corpus is small and topical enough that every top-5 chunk came from one of the 3 relevant papers on every query. **Reranking's real, measured benefit was recall**: on average only 2.4 of the 3 relevant papers appeared in the dense-only top-5 (one relevant paper sometimes got crowded out by chunks from the other two), while reranking recovered full 3-of-3 paper coverage on every query. This is a genuine, if modest, improvement from a no-training-required cross-encoder - worth keeping enabled by default in `pipeline.py`.

**LLM stance-label spot-check (2 examples):** mixed quality, an honest finding worth flagging before trusting generated explanations at face value.
- `right_hand` example: clean and accurate - correctly labeled the two genuinely relevant excerpts (explicitly describing the "motor mapping task" and "activation of the right hand") as SUPPORTS, and correctly marked three background/parcellation excerpts UNRELATED.
- `left_hand` example: the *retrieval-relevance* judgment was fine (all 5 chunks came from relevant papers, matching gold), but the model's **stated justification contained a factual error** - it described the SomMot network as involved in "working memory/cognitive control," which is actually the role of the *Cont* (frontoparietal control) network, not SomMot. The surface-level SUPPORTS/UNRELATED label was defensible, but the natural-language reasoning behind it was neuroscientifically wrong.

**Takeaway**: retrieval quality (now measured, not assumed) is solid at this corpus size, especially with reranking. Generation quality is more fragile - the 3B model can produce specific, confident-sounding neuroscience claims that are incorrect, even when its coarse relevance judgment is fine. Any real use of this pipeline's generated text should treat it as a retrieval-grounded *draft*, not a verified explanation, until stance-label accuracy is evaluated at more than 2-example scale.

Findings folded into [docs/case1-summary-report.md](../docs/case1-summary-report.md) §8.
